# 00 · Plan corregido: orientación antes de distancia
La revisión visual del primer intento encontró 57.43% como mezcla, solo dos columnas, y dos falsos planos de pared que eran acero del truss. DBSCAN agrupa estructuras conectadas: cambiar eps no basta para separarlas. Tampoco debemos decidir calidad por silhouette al comparar once grupos con cientos de segmentos.

| Hipótesis | Prueba | Criterio previo |
|---|---|---|
| H1: una pared debe ocupar superficie, no solo tener bbox grande | RANSAC; área ocupada y cobertura de grilla tangente | pared solo si área ocupada≥20 m² y cobertura≥55%; revisar candidatos rechazados y vista |
| H2: separar orientaciones rompe conexiones columna/viga | PCA local k=24/48/96; linealidad 0.30/0.45/0.60, planaridad 0.35/0.50/0.65 | medir distribución de orientaciones, instancias, mezcla y residuo; contrastar columnas en xy y lateral |
| H3: el crecimiento y la fusión reducen fragmentos | reasignación a <0.25/0.45/0.65 m; fusión colineal con hueco y distancia transversal acotados | evitar contar caras de una misma columna como objetos diferentes; curvas, cantidad de segmentos y posiciones |
| H4: orientación es el cambio decisivo | mismo pipeline con/sin orientación; DBSCAN y HDBSCAN, normales, color, height y voxel | mezcla<25%, ≥6 columnas; además ruido≤5% y mezcla+ruido<25%, para no esconder mezcla como ruido |
| H5: comparar en un espacio común y con control de K cambia la lectura | xyz+normales+linealidad estandarizados; segmentos≥500 puntos; soporte pareado; agregación de segmentos a K=11 | reportar silhouette/DB, cobertura, número de grupos y mayor segmento; no asumir que el filtro de tamaño elimina el sesgo por K |

Secuencia: 01 piso/techo y auditoría de pared; 02 sensibilidad de orientación y agrupación; 03 reglas honestas y color; 04 ablaciones, estabilidad y comparación; 05 propagación completa y revisión visual. La altura del piso sigue como unión de parches bajos a ≤0.40 m, con distancia <0.08 m. El eje e1 usa signo absoluto. Lineales verticales: |e1_z|>0.85; horizontales: |e1_z|<0.30, separados por dirección x/y. Planos se separan por normal vertical/horizontal; el resto es otro. PCA usa cKDTree solo sobre voxel.

Reglas físicas: columna si alto>2 m, alto/ancho≥3 y eje vertical>0.85; reportaremos además cuántas tienen alto≥5 m y sus posiciones para distinguir soportes parciales de columnas de la nave. Lineal horizontal de sección PCA aplanada (λ2/λ3>4) recibe viga_horizontal; sección ambigua o perfil inclinado recibe viga_o_tuberia. No llamaremos tubería a un perfil solo por baja planaridad/esfericidad: eso no prueba sección circular ni distingue tubo de perfil cerrado. No existe clase truss. Equipo bajo requiere extensión z<3, altura media sobre piso<3 y esfericidad≥0.03. Compacto plausible solo describe tamaño 0.4–2 m alto y 0.3–1 m ancho; no identifica persona/vehículo.

**Selección de la corrida final (incluye la enmienda de auditoría):** exigir piso fijo≥95%, mezcla<25%, ≥6 columnas, ruido≤5%, mezcla+ruido<25% y ≥6 columnas de alto≥5 m **en huellas xy separadas**. Para contar huellas, agrupamos centros xy a ≤1.5 m y contamos una sola por grupo; no altera etiquetas ni utiliza posiciones conocidas. Entre elegibles minimizar el número TOTAL de segmentos (penalizar fragmentación también por debajo de 500 puntos), después mezcla y después ARI de remuestreo descendente. Los controles sin orientación participan con el mismo resto de etapas. Si no hay elegibles, reportar la meta fallida y ordenar por mezcla+ruido, sin inventar cumplimiento. Las métricas de clustering se informan, no gobiernan esta selección. ARI/NMI entre clases finales colapsadas y los once grupos de clase solo son concordancia, no calidad ni exactitud.

La comparación geométrica usa un StandardScaler común en los 93 903 puntos de clase y linealidad calculada una vez con k=96. El soporte pareado excluye segmentos con <500 puntos en la nube completa de cualquiera de las dos particiones; se reporta cobertura y tamaño de ese soporte. Agregación a K=11: k-means sobre centroides geométricos de segmentos ponderados, conservando cada segmento entero. Es un control de resolución, no una segmentación física nueva. No se comparará silhouette sin informar K.

**Enmienda tras auditoría visual del primer ganador:** HDBSCAN100 tenía seis etiquetas altas pero solo cinco grupos de huellas, y faltaban columnas visibles alrededor de x≈3.5 y 13.5. La restricción de seis huellas distintas corrige ese conteo, sin cambiar métricas ni ajustar la segmentación a esas coordenadas. Se registra este cambio de selección, no se presenta como una regla fijada antes de la primera corrida.

Semilla fija 0. Estabilidad con tres réplicas voxel×{0.9,1,1.1} y retención aleatoria del 90%, evaluadas sobre la muestra común. Se repite RANSAC y PCA por réplica. Las tres comparaciones ARI no son intervalos de confianza. No hay ground truth.


**Ejecución:** IPython en `.venv`, celdas en orden, sin kernel de sockets (nbconvert bloqueado por sandbox). Tiempo: 0.00 s. Logs en `outputs/logs/`; cachés en `outputs/cache/`.